# Import Qwen

In [1]:
%load_ext autoreload

%autoreload 2

import warnings

# Ignore all FutureWarnings to clean up your notebook outputs
warnings.filterwarnings("ignore", category=FutureWarning)

from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import Image, display
from app.graph.workflow import build_graph
from app.core.app import container
from pprint import pprint as ppr
from app.memory.extractor import MemoryExtractor

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/nguyen/micromamba/envs/unsloth_fresh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17857.65it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11130.54it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents found: 32
[RAG] Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20132.82it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 6/6 [00:00<00:00,  6.61it/s]


Loading model: unsloth/Qwen3-4B-Instruct-2507-bnb-4bit
==((====))==  Unsloth 2026.5.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 7.603 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1271.75it/s]


unsloth/Qwen3-4B-Instruct-2507-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


# Test Graph Workflow

In [ ]:
graph = build_graph()

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is LangGraph, LangChain, LangSmith? And how people use them?",
            )
        ]
    }
)

display(Image(graph.get_graph().draw_mermaid_png()))

ppr(result)

Both `max_new_tokens` (=1024) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [3]:
extracted_memories = container.memory_extractor.extract(result["messages"])

Both `max_new_tokens` (=1024) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [4]:
extracted_memories

[Memory(id='d708ed7e-b940-4439-8b3c-66e95853b0f0', type=<MemoryType.PREFERENCE: 'preference'>, content='Refactor variable names only when they are ambiguous or misleading, prioritizing clarity and team consistency over style alone.', created_at=datetime.datetime(2026, 6, 27, 10, 44, 51, 639697, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 6, 27, 10, 44, 51, 639701, tzinfo=datetime.timezone.utc)),
 Memory(id='61f9ca9c-b3b3-4616-adfb-82f2621b998f', type=<MemoryType.FACT: 'fact'>, content='Variable name refactoring should follow team coding standards, such as camelCase or snake_case, to ensure consistency across the codebase.', created_at=datetime.datetime(2026, 6, 27, 10, 44, 51, 639709, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 6, 27, 10, 44, 51, 639710, tzinfo=datetime.timezone.utc)),
 Memory(id='6c517717-25a9-48ab-ae1d-cac591ae134c', type=<MemoryType.CONTEXT: 'context'>, content='Current context: Preparing to refactor variable names in a sh

In [5]:
# Save memories to table "memories"
container.memory_manager.memory_store.save(extracted_memories)

In [6]:
faiss_items = []

# Save mapping of memory and faiss_id to table "memory_faiss_mapping"
for memory in extracted_memories:

    # Get faiss_id
    faiss_id = (
        container.memory_manager.memory_store
        .get_next_faiss_id()
    )

    # Insert the mapping
    container.memory_manager.memory_store.save_faiss_mapping(
        faiss_id=faiss_id,
        memory_id=memory.id,
    )

    # Collect list of faiss id paired with is content
    faiss_items.append(
        (
            faiss_id,
            memory.content,
        )
    )

# Add all faiss items to the index
container.faiss_store.add_many(
    faiss_items
)

container.faiss_store.save()

# Any Tests

In [7]:
from datetime import UTC, datetime

print(datetime.now(UTC))

2026-06-23 15:20:50.664584+00:00


# Test Schemas

In [ ]:
from app.tools.registry import ToolRegistry
from pprint import pprint as ppr

registry = ToolRegistry(register_all_available=True)

ppr(registry.get_tool_schemas())


# Test Model generation

In [9]:
from app.core.app import container
from pprint import pprint as ppr
from langchain_core.messages import HumanMessage

ppr(container.model.invoke(
    [HumanMessage(content="What is the capital of Vietnam?")]
))

Both `max_new_tokens` (=1024) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgentDecision(thought='The capital of Vietnam is a straightforward factual question.', response='Hanoi', tool_calls=[])
